In [1]:
import pandas as pd
import numpy as np

deliveries = pd.read_csv('../data/raw/deliveries.csv')
matches = pd.read_csv('../data/raw/matches.csv')

# Filter 2nd innings only
inn2 = deliveries[deliveries['inning'] == 2].copy()

# Merge with matches
df = inn2.merge(
    matches[['id', 'winner', 'target_runs']],
    left_on='match_id', right_on='id'
)

# Label: did batting team win?
df['batting_team_won'] = (df['batting_team'] == df['winner']).astype(int)

# Sort by match and ball order
df = df.sort_values(['match_id', 'over', 'ball']).reset_index(drop=True)

print("Shape:", df.shape)
print("Done!")

Shape: (125741, 21)
Done!


In [2]:
# Cumulative runs scored so far
df['cumulative_runs'] = df.groupby('match_id')['total_runs'].cumsum()

# Cumulative wickets fallen
df['wickets_fallen'] = df.groupby('match_id')['is_wicket'].cumsum()
df['wickets_in_hand'] = 10 - df['wickets_fallen']

# Ball number (1 to 120)
df['ball_number'] = (df['over'] * 6) + df['ball']
df['balls_remaining'] = 120 - df['ball_number']

# Runs required
df['runs_required'] = df['target_runs'] - df['cumulative_runs']

# Run rates
df['crr'] = (df['cumulative_runs'] / df['ball_number']) * 6
df['rrr'] = (df['runs_required'] / df['balls_remaining'].replace(0, 1)) * 6
df['rrr_crr_diff'] = df['rrr'] - df['crr']

print("Basic features done!")
print(df[['cumulative_runs','wickets_in_hand','balls_remaining',
          'runs_required','crr','rrr']].head(10))

Basic features done!
   cumulative_runs  wickets_in_hand  balls_remaining  runs_required       crr  \
0                1               10              119          222.0  6.000000   
1                2               10              118          221.0  6.000000   
2                2               10              117          221.0  4.000000   
3                3               10              116          220.0  4.500000   
4                4               10              115          219.0  4.800000   
5                4               10              114          219.0  4.000000   
6                4               10              113          219.0  3.428571   
7                4                9              113          219.0  3.428571   
8                4                9              112          219.0  3.000000   
9                8                9              111          215.0  5.333333   

         rrr  
0  11.193277  
1  11.237288  
2  11.333333  
3  11.379310  
4  11.426087

In [3]:
# Match phase
def get_phase(over):
    if over <= 5: return 0    # Powerplay
    elif over <= 14: return 1  # Middle overs
    else: return 2             # Death overs

df['phase'] = df['over'].apply(get_phase)

# Runs in last 5 overs (momentum signal)
df['last_5_overs_runs'] = df.groupby('match_id')['total_runs']\
    .transform(lambda x: x.rolling(30, min_periods=1).sum())

# Partnership runs (runs since last wicket)
df['partnership_key'] = df.groupby('match_id')['wickets_fallen'].transform(lambda x: x)
df['partnership_runs'] = df.groupby(
    ['match_id','partnership_key']
)['total_runs'].cumsum()

# Venue average 2nd innings score
venue_avg = matches.groupby('venue')['target_runs'].mean().reset_index()
venue_avg.columns = ['venue', 'venue_avg']
df = df.merge(matches[['id','venue']], left_on='match_id', right_on='id')
df = df.merge(venue_avg, on='venue')

print("All features done!")
print(df[['phase','last_5_overs_runs','partnership_runs','venue_avg']].head(10))

All features done!
   phase  last_5_overs_runs  partnership_runs  venue_avg
0      0                1.0                 1  167.34375
1      0                2.0                 2  167.34375
2      0                2.0                 2  167.34375
3      0                3.0                 3  167.34375
4      0                4.0                 4  167.34375
5      0                4.0                 4  167.34375
6      0                4.0                 4  167.34375
7      0                4.0                 0  167.34375
8      0                4.0                 0  167.34375
9      0                8.0                 4  167.34375


In [4]:
# Remove impossible rows
df = df[df['balls_remaining'] > 0]
df = df[df['runs_required'] > 0]

# Final feature list
FEATURES = [
    'runs_required', 'balls_remaining', 'wickets_in_hand',
    'rrr', 'crr', 'rrr_crr_diff', 'phase',
    'last_5_overs_runs', 'partnership_runs', 'venue_avg'
]

# Save to processed folder
final_df = df[FEATURES + ['match_id', 'batting_team_won']]
final_df.to_csv('../data/processed/model_data.csv', index=False)

print("Saved successfully!")
print("Final shape:", final_df.shape)
print("\nFeature summary:")
print(final_df[FEATURES].describe().round(2))

Saved successfully!
Final shape: (124582, 12)

Feature summary:
       runs_required  balls_remaining  wickets_in_hand        rrr        crr  \
count      124582.00        124582.00        124582.00  124582.00  124582.00   
mean           94.56            63.28             7.53      10.83       7.58   
std            50.51            33.16             2.14      13.37       2.36   
min             1.00             1.00             0.00       0.07       0.00   
25%            55.00            35.00             6.00       7.28       6.36   
50%            93.00            64.00             8.00       9.03       7.59   
75%           132.00            92.00             9.00      11.21       8.85   
max           287.00           119.00            10.00     792.00      36.00   

       rrr_crr_diff      phase  last_5_overs_runs  partnership_runs  venue_avg  
count     124582.00  124582.00          124582.00         124582.00  124582.00  
mean           3.25       0.87              33.63    